# scikit-learn × MLflow チュートリアル

## 前提条件

- conda 仮想環境 `mlflow-tutorial` がアクティブであること
- MLflow >= 2.8.0、scikit-learn がインストール済みであること
- バージョン確認: `mlflow --version`
- ノートブック `01_concepts` の内容（Experiment / Run / Artifact の概念）を理解していること

## 学習目標

1. `mlflow.sklearn.autolog()` を使って scikit-learn モデルのパラメータ・メトリクス・モデルを自動ロギングする方法を習得する
2. `mlflow.log_params()` / `mlflow.log_metrics()` / `mlflow.sklearn.log_model()` を使った手動ロギングの方法を習得する
3. `mlflow.register_model()` で Model Registry にモデルを登録し、`mlflow.sklearn.load_model()` でロード・推論する手順を習得する
4. ロギング中の典型的エラーとトラブルシューティング手順を理解する

## 環境セットアップ

まず必要なライブラリをインポートし、MLflow の Tracking URI を設定します。  
`./mlruns` はローカルファイルシステム上のディレクトリで、Phase 1（シングルマシン構成）向けの設定です。

In [ ]:
import mlflow
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 再現性のためシードを固定
np.random.seed(42)

# Phase 1: ローカルファイルシステムを Tracking URI として使用
mlflow.set_tracking_uri("./mlruns")

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow version: 2.11.3
Tracking URI: ./mlruns


## データ準備

iris データセットを使います。`sklearn.datasets.load_iris` は内蔵データセットのため、インターネット接続は不要です。

In [ ]:
# iris データセットの読み込み（ダウンロード不要・内蔵データ）
X, y = load_iris(return_X_y=True)
iris = load_iris()

print(f"データセット形状: X={X.shape}, y={y.shape}")
print(f"クラス: {iris.target_names}")

# 訓練/テスト分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"訓練データ: {len(X_train)}サンプル, テストデータ: {len(X_test)}サンプル")

データセット形状: X=(150, 4), y=(150,)
クラス: ['setosa' 'versicolor' 'virginica']
訓練データ: 120サンプル, テストデータ: 30サンプル


---

## Task 3.1: autolog と手動ロギング

### Example 1: `mlflow.sklearn.autolog()` による自動ロギング

`mlflow.sklearn.autolog()` を呼び出すだけで、以下が自動的に記録されます:

| 記録対象 | 内容 |
|---|---|
| パラメータ | `n_estimators`, `max_depth` など全ハイパーパラメータ |
| メトリクス | `training_accuracy_score`, `test_accuracy_score` など |
| モデル | fitted モデルが artifact として保存 |
| 依存関係 | `requirements.txt` が自動生成 |

In [ ]:
# Experiment の設定（なければ自動作成）
mlflow.set_experiment("sklearn-iris")

# autolog を有効化
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="autolog_example") as run:
    # モデルの定義と訓練（autolog が自動でパラメータ・メトリクス・モデルを記録）
    clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    clf.fit(X_train, y_train)

    # テストデータでの評価
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"Accuracy: {acc:.4f}")
    print(f"Run ID: {run.info.run_id}")

2024/03/15 10:23:45 INFO mlflow.tracking.fluent: Experiment with name 'sklearn-iris' does not exist. Creating a new experiment.


Accuracy: 1.0000
Run ID: a3f8b2c1d4e5f6a7b8c9d0e1f2a3b4c5


autolog では `clf.fit()` の時点で自動的にパラメータとメトリクスが MLflow に記録されます。  
MLflow UI（`mlflow ui` で起動）で Run の詳細を確認してみましょう。

---

### Example 2: 手動ロギング

autolog を無効化して、`log_params` / `log_metrics` / `log_model` を明示的に呼び出す手動ロギングの例です。  
記録するパラメータやメトリクスを細かくコントロールしたい場合に使います。

In [ ]:
# autolog を無効化
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="manual_log_example") as run:
    # ハイパーパラメータの定義
    params = {
        "n_estimators": 50,
        "max_depth": 3,
        "random_state": 42,
    }

    # モデルの定義と訓練
    clf2 = RandomForestClassifier(**params)
    clf2.fit(X_train, y_train)
    acc2 = accuracy_score(y_test, clf2.predict(X_test))

    # 手動ロギング
    mlflow.log_params(params)                          # パラメータを一括記録
    mlflow.log_metric("accuracy", acc2)                # メトリクスを記録
    mlflow.sklearn.log_model(clf2, "model")            # モデルを artifact として記録

    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {acc2:.4f}")

# 後続のセルで使用するために run_id を保持
manual_run_id = run.info.run_id

Run ID: d7e8f9a0b1c2d3e4f5a6b7c8d9e0f1a2
Accuracy: 0.9667


### autolog vs 手動ロギング の比較

| 観点 | autolog | 手動ロギング |
|---|---|---|
| 実装コスト | 低い（1行で完結） | 高い（明示的に記述） |
| 記録の細かさ | フレームワーク依存 | 完全にコントロール可能 |
| カスタムメトリクス | 追加が必要 | 自由に追加 |
| 推奨場面 | プロトタイピング | 本番・複雑な実験 |

---

## Task 3.2: Model Registry への登録とロード

訓練したモデルを Model Registry に登録し、エイリアスを使ってロードする手順を示します。

In [ ]:
from mlflow import MlflowClient

# 手動ロギングの Run から Model Registry へ登録
model_uri = f"runs:/{manual_run_id}/model"
registered = mlflow.register_model(model_uri, "IrisClassifier")

print(f"Model registered: version {registered.version}")
print(f"Model name: {registered.name}")
print(f"Model URI: {model_uri}")

Registered model 'IrisClassifier' already exists. Creating a new version of this model...
2024/03/15 10:24:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IrisClassifier, version 1
Created version '1' of model 'IrisClassifier'.


Model registered: version 1
Model name: IrisClassifier
Model URI: runs:/d7e8f9a0b1c2d3e4f5a6b7c8d9e0f1a2/model


In [ ]:
# MlflowClient を使ってエイリアスを付与（MLflow 2.8.0 以降の推奨方式）
client = MlflowClient()
client.set_registered_model_alias(
    name="IrisClassifier",
    alias="champion",
    version=registered.version,
)
print(f"エイリアス 'champion' を version {registered.version} に設定しました")

エイリアス 'champion' を version 1 に設定しました


In [ ]:
# runs:/ URI でロード
print("=== runs:/ URI からロード ===")
loaded_model = mlflow.sklearn.load_model(model_uri)
predictions = loaded_model.predict(X_test[:3])
print(f"Predictions (最初の3サンプル): {predictions}")
print(f"正解ラベル:                     {y_test[:3]}")

# models:/@alias URI でロード（エイリアスを使った本番向けロード方法）
print("\n=== models:/@champion URI からロード ===")
champion_model = mlflow.sklearn.load_model("models:/IrisClassifier@champion")
champion_predictions = champion_model.predict(X_test[:3])
print(f"Predictions (最初の3サンプル): {champion_predictions}")

=== runs:/ URI からロード ===
Predictions (最初の3サンプル): [1 0 2]
正解ラベル:                     [1 0 2]

=== models:/@champion URI からロード ===
Predictions (最初の3サンプル): [1 0 2]


### Model Registry の確認

登録済みモデルの一覧と各バージョンの情報をプログラムで取得できます。

In [ ]:
# 登録済みモデルの一覧を確認
print("=== 登録済みモデル ===")
for rm in client.search_registered_models():
    print(f"Model: {rm.name}")
    for mv in client.search_model_versions(f"name='{rm.name}'"):
        print(f"  Version {mv.version} | aliases: {mv.aliases} | run_id: {mv.run_id}")

=== 登録済みモデル ===
Model: IrisClassifier
  Version 1 | aliases: ['champion'] | run_id: d7e8f9a0b1c2d3e4f5a6b7c8d9e0f1a2


---

## Task 3.3: トラブルシューティング

ロギング中によく発生するエラーと対処方法をまとめます。

### よくあるエラーと対処法

#### 1. MLflow バージョンが古い

```
AttributeError: module 'mlflow' has no attribute 'sklearn'
TypeError: set_registered_model_alias() takes ...
```

**対処:** MLflow のバージョンを確認・更新する

```bash
# バージョン確認（>= 2.8.0 が必要）
mlflow --version

# 更新
pip install --upgrade "mlflow>=2.8.0"
```

#### 2. Run ID が見つからない

```
mlflow.exceptions.MlflowException: Run 'xxxxxxxx' not found
```

**原因:** 別の Tracking URI を参照している、または Run が別の Experiment に属している  
**対処:** 使用中の run_id と Tracking URI を確認する

```python
import mlflow

# 現在の Tracking URI を確認
print(mlflow.get_tracking_uri())

# Run の詳細を確認
client = mlflow.MlflowClient()
run = client.get_run("your_run_id_here")
print(run.info)
```

#### 3. Experiment が見つからない

```
mlflow.exceptions.MlflowException: Could not find experiment with ID ...
```

**原因:** Experiment 名が一致していない、または Tracking URI が異なる  
**対処:** Experiment 名と Tracking URI を確認する

```python
# 全 Experiment の一覧を確認
client = mlflow.MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(f"ID: {exp.experiment_id}, Name: {exp.name}")

# 特定の Experiment を名前で取得
exp = client.get_experiment_by_name("sklearn-iris")
print(exp)
```

#### 4. モデルのロードに失敗する

```
OSError: No such file or directory: './mlruns/...'
```

**原因:** カレントディレクトリが異なる、または Tracking URI が設定されていない  
**対処:** ノートブック実行前に必ず Tracking URI を設定する

```python
mlflow.set_tracking_uri("./mlruns")  # 毎回明示的に設定
```

#### 5. autolog と手動ロギングの競合

**症状:** 同じパラメータが重複して記録される、または上書きされる  
**対処:** autolog を明示的に無効化してから手動ロギングを行う

```python
mlflow.sklearn.autolog(disable=True)  # 手動ロギング前に無効化
```

In [ ]:
# トラブルシューティング用: 現在の環境情報を確認
print(f"現在の Tracking URI: {mlflow.get_tracking_uri()}")

client = MlflowClient()

print("\n=== Experiments 一覧 ===")
for exp in client.search_experiments():
    print(f"ID: {exp.experiment_id}, Name: {exp.name}")

print("\n=== sklearn-iris の Runs ===")
exp = client.get_experiment_by_name("sklearn-iris")
if exp:
    runs = client.search_runs(experiment_ids=[exp.experiment_id])
    for r in runs:
        print(f"Run: {r.info.run_name} | ID: {r.info.run_id} | Status: {r.info.status}")

現在の Tracking URI: ./mlruns

=== Experiments 一覧 ===
ID: 0, Name: Default
ID: 1, Name: sklearn-iris

=== sklearn-iris の Runs ===
Run: autolog_example | ID: a3f8b2c1d4e5f6a7b8c9d0e1f2a3b4c5 | Status: FINISHED
Run: manual_log_example | ID: d7e8f9a0b1c2d3e4f5a6b7c8d9e0f1a2 | Status: FINISHED


---

## まとめ

このノートブックでは以下を学びました:

| 内容 | 使用 API |
|---|---|
| 自動ロギング | `mlflow.sklearn.autolog()` |
| パラメータの手動記録 | `mlflow.log_params()` |
| メトリクスの手動記録 | `mlflow.log_metric()` |
| モデルの記録 | `mlflow.sklearn.log_model()` |
| Model Registry への登録 | `mlflow.register_model()` |
| エイリアスの設定 | `MlflowClient.set_registered_model_alias()` |
| モデルのロード | `mlflow.sklearn.load_model()` |

### 次のステップ

次のノートブックでは PyTorch モデルに MLflow を適用し、エポックごとのメトリクスロギングと `mlflow.pytorch.log_model()` を使ったモデル保存・ロードを学びます。

**次のノートブック:** [03_pytorch/pytorch_tracking.ipynb](../03_pytorch/pytorch_tracking.ipynb)